# Rename L3D++_data，从 img_name 命名改为 img_id

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import random
import networkx as nx
import matplotlib.pyplot as plt

from datasets.dataset_reader import load_sparse_model


def main_colmap():
    ####################################### 参数 #######################################
    sparse_model_path = r"/home/rylynn/Pictures/LinesDetection_Workspace/datasets/Dublin_block3/sparse_txt/"
    line2d_path = r"/home/rylynn/Pictures/LinesDetection_Workspace/output/dublin_block3_md_resize4/L3D++_data/"
    camerasInfo, _ = load_sparse_model(sparse_model_path, image_scale=1)
    for cam_id, cam_dict in enumerate(camerasInfo):
        width = int(cam_dict['width'])
        height = int(cam_dict['height'])
        img_name = cam_dict['img_name'].split('/')[-1]
        line2d_filename1 = f'segments_L3D++_{img_name}_{width}x{height}.bin'
        cam_id_new = cam_id+1 # cam_id 从0开始，img_id从1开始
        line2d_filename2 = f'segments_L3D++_{cam_id_new}_{width}x{height}.bin'
        old_path = os.path.join(line2d_path, line2d_filename1)
        new_path = os.path.join(line2d_path, line2d_filename2)
        os.rename(old_path, new_path)
        print(f"{img_name}更名成功")


if __name__ == "__main__":
    main_colmap()

# 从L3D++_data中选择长度前3000的线段

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import random
import networkx as nx
import matplotlib.pyplot as plt
import cv2

from datasets.dataset_reader import (
    load_sparse_model,
    match_pair,
    read_depth
)
from datasets.line3dpp_loader import parse_line_segments, parse_lines3dpp, save_segments_l3dpp
from utils.visualize import viz_lines2D2

if __name__ == "__main__":
    ####################################### 参数 #######################################
    workspace = r"/home/rylynn/Pictures/LinesDetection_Workspace/datasets/group611_800z"

    ####################################### 路径 #######################################
    sparse_model_path = os.path.join(workspace, 'sparse_txt')
    images_path = os.path.join(workspace, 'images')
    output_path = '/home/rylynn/Pictures/LinesDetection_Workspace/datasets/group611_800z/sparse_txt_geo/'
    lsd_lines_path = os.path.join(output_path, 'Line3D++_H')
    len3000_path = os.path.join(output_path, 'Line3D++')
    os.makedirs(len3000_path, exist_ok=True)
    viz_path = os.path.join(len3000_path, 'viz_lsd_len3000')
    os.makedirs(viz_path, exist_ok=True)
    

    # 0. 数据准备：读取稀疏模型，读取LSD检测所有线段
    camerasInfo, points_in_images = load_sparse_model(sparse_model_path, image_scale=1)
    print(f"[INFO] Loaded {len(camerasInfo)} images.")

    # 1. 取长度前3000的线段存储为Line3D++的输入
    topk_indices_all = {}
    for img_id, cam_dict in tqdm(enumerate(camerasInfo), total=len(camerasInfo), desc="Processing lines"):
        cam_dict = camerasInfo[img_id]
        width = int(cam_dict['width'])
        height = int(cam_dict['height'])
        img_name = cam_dict['img_name'].split('/')[-1]

        # 0205:预先把空的文件删除了，所以这里加个判断
        filename = 'segments_L3D++_'+ str(img_id+1) + '_' + str(width) + 'x' + str(height) + '.bin'
        filepath = os.path.join(lsd_lines_path, 'L3D++_data', filename)
        if not os.path.exists(filepath):
            print(f"[WARNING] File {filepath} does not exist. Skipping.")
            continue

        lines = parse_line_segments(lsd_lines_path, img_id+1, width, height)[:, [1,0,3,2]]


        # 取长度前3000的线段
        lines_lengths = ((lines[:, 0]-lines[:, 2])**2 + (lines[:, 1]-lines[:, 3])**2)**0.5

        # 对长度进行降序排序，获取对应的原始索引
        # argsort 返回的是：原本在哪个位置的元素现在排在这里
        sorted_indices = np.argsort(-lines_lengths)
        # 取前 3000 个索引
        top_k = min(3000, lines.shape[0])  # 防止 lines 总数不足 3000
        topk_indices = sorted_indices[:top_k]
        # 根据索引提取对应的线段
        lines_topk = lines[topk_indices]
        topk_indices_all[img_id] = topk_indices.tolist()
 
        save_segments_l3dpp(lines_topk, len3000_path, img_id+1, width, height)
        #img = cv2.imread(os.path.join(images_path, cam_dict['img_name']+'.png'), 0)
        #viz_lines2D2(img, lines_topk[:, [1,0,3,2]].reshape(-1,2,2), viz_path, f"{os.path.splitext(img_name)[0]}")
    
    # 保存top3000线段的索引
    #with open(os.path.join(len3000_path, 'top3000_indices.json'), 'w') as f:
        #json.dump(topk_indices_all, f)


# convert xml(Line3D++) to bin(ISPRS Journal2024)

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import random
import networkx as nx
import matplotlib.pyplot as plt
import struct
import re
import math

from datasets.dataset_reader import (
    load_sparse_model,
    match_pair,
    read_depth
)
from datasets.line3dpp_loader import parse_line_segments, parse_lines3dpp, save_segments_l3dpp
from utils.visualize import viz_lines2D2


def convert_xml_to_opencv_bin(input_xml_path, output_bin_path):
    """
    读取XML文件，转换为C++ OpenCV可读的二进制文件
    每行包含7个浮点数: x1, y1, x2, y2, cx, cy, length
    """
    with open(input_xml_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 提取线段端点坐标
    pattern = re.compile(
        r'<x>([0-9eE\+\-\.]+)</x>\s*'
        r'<y>([0-9eE\+\-\.]+)</y>\s*'
        r'<z>([0-9eE\+\-\.]+)</z>\s*'
        r'<w>([0-9eE\+\-\.]+)</w>'
    )

    matches = pattern.findall(content)
    filename = os.path.basename(input_xml_path)

    if not matches:
        print(f"[跳过] {filename}: 未在文件中找到线段数据！")
        return False

    num_lines = len(matches)
    
    # 设置矩阵格式：N行 x 7列 的单通道矩阵 (CV_32F, type=5)
    rows = num_lines
    cols = 7  # 每行7个数字
    mat_type = 5  # CV_32F (单通道浮点)

    with open(output_bin_path, 'wb') as f:
        # 写入头部 (3个整型: rows, cols, type)，使用小端序 '<i'
        f.write(struct.pack('<i', rows))
        f.write(struct.pack('<i', cols))
        f.write(struct.pack('<i', mat_type))

        # 写入矩阵数据：每行7个浮点数
        for match in matches:
            x1, y1, x2, y2 = map(float, match)
            
            # 计算中心点和长度
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0
            length = math.sqrt((x1 - x2) * (x1 - x2) + (y1 - y2) * (y1 - y2))
            
            # 按顺序写入7个浮点数
            f.write(struct.pack('<f', x1))
            f.write(struct.pack('<f', y1))
            f.write(struct.pack('<f', x2))
            f.write(struct.pack('<f', y2))
            f.write(struct.pack('<f', cx))
            f.write(struct.pack('<f', cy))
            f.write(struct.pack('<f', length))

    print(f"[成功] {filename}: 转换了 {num_lines} 条线段，格式 {rows}x{cols} CV_32F")
    return True


if __name__ == "__main__":
    ####################################### 参数 #######################################
    workspace = r"/home/rylynn/Pictures/LinesDetection_Workspace/datasets/Dublin_block3/"

    ####################################### 路径 #######################################
    sparse_model_path = os.path.join(workspace, 'sparse_txt')
    images_path = os.path.join(workspace, 'images')
    output_path = '/home/rylynn/Pictures/LinesDetection_Workspace/output/dublin_block3_md_resize4/L3D++_data_resized/'
    input_lines_path = os.path.join(output_path, 'L3D++_data')
    output_lines_path = os.path.join(output_path, 'L3D++_data_bin')
    os.makedirs(output_lines_path, exist_ok=True)
    
    camerasInfo, points_in_images = load_sparse_model(sparse_model_path, image_scale=1)
    print(f"[INFO] Loaded {len(camerasInfo)} images.")

    for img_id, cam_dict in enumerate(camerasInfo):
        cam_dict = camerasInfo[img_id]
        width = int(cam_dict['width'])
        height = int(cam_dict['height'])
        img_name = cam_dict['img_name'].split('/')[-1]

        input_xml_path = os.path.join(input_lines_path, f'segments_L3D++_{img_id+1}_{width}x{height}.bin')
        output_xml_path = os.path.join(output_lines_path, f'{img_name}.jpg.bin')

        convert_xml_to_opencv_bin(input_xml_path, output_xml_path)
